# NB05 — Colour Shortcut

Quantifies how much of BDLitchi's classification accuracy is attributable to colour
and background statistics rather than lesion morphology, motivated by the observation
that a 32-bin HSV-histogram 5-NN classifier reaches 77.09% accuracy on 11 classes
(chance ≈ 9%) using colour alone. Three complementary probes:

1. Colour-histogram kNN, per class, on the clean `common_test` set — identifies which
   classes are solvable by colour alone.
2. A grayscale-trained CNN (ResNet-50, colour channel removed) — the accuracy drop
   relative to the colour-trained counterpart is a direct estimate of the model's
   reliance on colour.
3. A background-suppressed (centre-crop) variant — a low-cost probe of background
   reliance.

External validation (zero-shot evaluation on an independent litchi dataset, its
class-merging documentation, and the before/after-deduplication comparison) is a
separate notebook, NB11, so it can be run and reused independently of the
grayscale-training step here.

**Runtime:** ≈ 1.5–2.5 h (one grayscale training run plus colour-kNN feature extraction).


In [1]:
# ===== Imports =====
import os, json, time, random, warnings, math
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
import timm

from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, matthews_corrcoef, confusion_matrix, roc_auc_score)

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')


device: cuda | Tesla T4


In [2]:
# ===== Config =====
RAW_DATASET_DIR = '/kaggle/input/datasets/maruf170102/bdlithi/Dataset'   # <-- EDIT: folder containing the 11 class sub-folders
SPLITS_DIR   = Path('/kaggle/input/datasets/maruf170102/bdlitchi-revision-splits/revision_splits')      # NB01 output
OUT_DIR      = Path('/kaggle/working/revision_shortcut'); OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED=42; EPOCHS=30; PATIENCE=7
GRAY_MODEL='ResNet50'

# --- colour reference checkpoint -------------------------------------------------
# The grayscale model below is trained on the GROUP split, so its colour counterpart
# must also be group-trained or the comparison is meaningless. Both NB02 and NB03
# produce a suitable ResNet-50; list whichever dataset(s) you have attached and the
# first match wins. NB02's ablation output is the usual one to have first.
CKPT_SEARCH_DIRS = [
    Path('/kaggle/input/datasets/maruf170102/nb02-corrected-leakage-ablation/revision_ablation'),   # <-- EDIT to your NB02 output slug
    # Path('/kaggle/input/bdlitchi-revision-seeds-cnn'),  # <-- NB03 output, if it has finished
]
# Ordered by preference. Group-trained only - never the naive-trained checkpoint.
CKPT_NAMES = [
    f'ResNet50_group_seed{SEED}.pt',   # from NB02 (trained on the group split)
    f'ResNet50_seed{SEED}.pt',         # from NB03 (also group split)
]

def _resolve_ref_ckpt():
    for d in CKPT_SEARCH_DIRS:
        for n in CKPT_NAMES:
            for cand in (Path(d)/'checkpoints'/n, Path(d)/n):
                if cand.exists(): return cand
    return None

EVAL_CKPT = _resolve_ref_ckpt()
if EVAL_CKPT:
    print('colour reference checkpoint:', EVAL_CKPT)
else:
    print('*** No colour reference checkpoint found. Searched:')
    for d in CKPT_SEARCH_DIRS: print('   ', d)
    print('    The grayscale training and colour kNN will still run, but the')
    print('    colour-vs-grayscale comparison (concern #8) will be SKIPPED.')
    print('    Attach your NB02 or NB03 output and fix CKPT_SEARCH_DIRS above.')
    import os
    if Path('/kaggle/input').exists():
        print('    Currently attached:', os.listdir('/kaggle/input'))

# --- reuse a previous NB05 run -------------------------------------------------
# Kaggle 'Save Version' reruns every cell, so re-running from scratch costs ~90 min
# to redo the kNN features and the grayscale training. Attach your previous NB05
# output as a dataset and set this to reuse those results.
# Leave as None to recompute everything from scratch.
PREV_RUN_DIR = None   # e.g. Path('/kaggle/input/nb05-colour-shortcut/revision_shortcut')


colour reference checkpoint: /kaggle/input/datasets/maruf170102/nb02-corrected-leakage-ablation/revision_ablation/checkpoints/ResNet50_group_seed42.pt


In [3]:
def set_all_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [4]:
IMAGENET_MEAN=[0.485,0.456,0.406]; IMAGENET_STD=[0.229,0.224,0.225]

class LitchiDataset(Dataset):
    def __init__(self, df, transform, label2idx, grayscale=False):
        self.df=df.reset_index(drop=True); self.t=transform
        self.l2i=label2idx; self.gray=grayscale
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; img=Image.open(r.filepath).convert('RGB')
        if self.gray:
            img = img.convert('L').convert('RGB')   # drop colour, keep 3 channels
        return self.t(img), self.l2i[r.label]

def make_transforms(sz, augment):
    if augment:
        tr=transforms.Compose([transforms.Resize((sz+32,sz+32)), transforms.RandomCrop(sz),
            transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
            transforms.ColorJitter(0.3,0.3,0.2,0.05), transforms.RandomRotation(20),
            transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)])
    else:
        tr=transforms.Compose([transforms.Resize((sz,sz)), transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)])
    ev=transforms.Compose([transforms.Resize((sz,sz)), transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)])
    return tr,ev

def build_backbone(name, nc, pretrained=True):
    if name=='ResNet50':
        m=torchvision.models.resnet50(weights='IMAGENET1K_V2' if pretrained else None)
        m.fc=nn.Linear(m.fc.in_features,nc); return m,224,32
    if name=='MobileNetV3Large':
        m=torchvision.models.mobilenet_v3_large(weights='IMAGENET1K_V2' if pretrained else None)
        m.classifier[-1]=nn.Linear(m.classifier[-1].in_features,nc); return m,224,32
    if name=='EfficientNetB0':
        return timm.create_model('efficientnet_b0',pretrained=pretrained,num_classes=nc),224,32
    if name=='EfficientNetB3':
        return timm.create_model('efficientnet_b3',pretrained=pretrained,num_classes=nc),300,16
    if name=='ViTBase16':
        return timm.create_model('vit_base_patch16_224',pretrained=pretrained,num_classes=nc),224,16
    raise ValueError(name)


In [5]:
def train_model(name, trn_df, val_df, label2idx, seed, epochs=30, patience=7,
                lr=3e-4, wd=1e-4, grayscale=False, log_every=1):
    """Independent training run. Returns (model, history, best_epoch)."""
    set_all_seeds(seed)
    nc=len(label2idx)
    model,sz,bs = build_backbone(name,nc); model=model.to(device)
    ttf,etf = make_transforms(sz, augment=True)
    trn=DataLoader(LitchiDataset(trn_df,ttf,label2idx,grayscale), batch_size=bs, shuffle=True,
                   num_workers=2, pin_memory=True, drop_last=False)
    val=DataLoader(LitchiDataset(val_df,etf,label2idx,grayscale), batch_size=bs, shuffle=False,
                   num_workers=2, pin_memory=True)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=wd)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    crit=nn.CrossEntropyLoss()
    scaler=torch.cuda.amp.GradScaler(enabled=(device.type=='cuda'))
    hist={'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}
    best=float('inf'); best_state=None; best_ep=-1; bad=0
    for ep in range(epochs):
        model.train(); rl=rc=rt=0
        for x,y in trn:
            x,y=x.to(device,non_blocking=True),y.to(device,non_blocking=True)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
                o=model(x); loss=crit(o,y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            rl+=loss.item()*x.size(0); rc+=(o.argmax(1)==y).sum().item(); rt+=x.size(0)
        tl,ta=rl/rt,rc/rt
        model.eval(); vl=vc=vt=0
        with torch.no_grad():
            for x,y in val:
                x,y=x.to(device),y.to(device)
                with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
                    o=model(x); loss=crit(o,y)
                vl+=loss.item()*x.size(0); vc+=(o.argmax(1)==y).sum().item(); vt+=x.size(0)
        vl,va=vl/vt,vc/vt
        sch.step()
        hist['train_loss'].append(tl); hist['val_loss'].append(vl)
        hist['train_acc'].append(ta);  hist['val_acc'].append(va)
        if log_every and (ep+1)%log_every==0:
            print(f'  ep {ep+1:02d}/{epochs} train_loss={tl:.4f} acc={ta:.4f} | val_loss={vl:.4f} acc={va:.4f}')
        if vl<best-1e-5:
            best=vl; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            best_ep=ep+1; bad=0
        else:
            bad+=1
            if bad>=patience:
                print(f'  early stop @ epoch {ep+1} (best epoch {best_ep})'); break
    model.load_state_dict(best_state)
    return model, hist, best_ep

@torch.no_grad()
def predict(model, df, label2idx, img_size, bs=32, grayscale=False):
    _,etf=make_transforms(img_size,False)
    ld=DataLoader(LitchiDataset(df,etf,label2idx,grayscale),batch_size=bs,shuffle=False,num_workers=2)
    model.eval(); P=[];Y=[];B=[]
    for x,y in ld:
        p=F.softmax(model(x.to(device)),1).cpu().numpy()
        P.extend(p.argmax(1)); Y.extend(y.numpy()); B.extend(p)
    return np.array(Y),np.array(P),np.array(B)

def metrics_dict(y,p,prob,nc):
    d=dict(accuracy=accuracy_score(y,p), balanced_accuracy=balanced_accuracy_score(y,p),
           f1_macro=f1_score(y,p,average='macro',zero_division=0),
           f1_weighted=f1_score(y,p,average='weighted',zero_division=0),
           precision_macro=precision_score(y,p,average='macro',zero_division=0),
           recall_macro=recall_score(y,p,average='macro',zero_division=0),
           mcc=matthews_corrcoef(y,p))
    try: d['roc_auc_macro']=roc_auc_score(np.eye(nc)[y],prob,average='macro',multi_class='ovr')
    except Exception: d['roc_auc_macro']=float('nan')
    return d


In [6]:
# ===== Artifact saving: predictions, probabilities, histories, calibration =====
def expected_calibration_error(y_true, y_prob, n_bins=15):
    conf = y_prob.max(1); pred = y_prob.argmax(1); acc = (pred==y_true).astype(float)
    bins=np.linspace(0,1,n_bins+1); ece=0.0; rows=[]
    for i in range(n_bins):
        m=(conf>bins[i])&(conf<=bins[i+1])
        if m.sum()==0: rows.append((float(bins[i]),float(bins[i+1]),0,np.nan,np.nan)); continue
        a,c=acc[m].mean(),conf[m].mean()
        ece+=(m.sum()/len(conf))*abs(a-c)
        rows.append((float(bins[i]),float(bins[i+1]),int(m.sum()),float(a),float(c)))
    return float(ece), rows

def save_run_artifacts(out_dir, tag, y_true, y_pred, y_prob, history=None,
                       classes=None, extra=None, specimen_ids=None, filenames=None):
    """Everything NB06 needs, with no re-inference.

    specimen_ids is essential: NB06 bootstraps by SPECIMEN, not by image. Images inside
    one specimen are the same leaf and are strongly correlated, so an image-level
    bootstrap would badly understate the confidence interval.
    """
    pdir=Path(out_dir)/'predictions'; pdir.mkdir(parents=True, exist_ok=True)
    payload=dict(y_true=np.asarray(y_true), y_pred=np.asarray(y_pred), y_prob=np.asarray(y_prob))
    if specimen_ids is not None:
        payload['specimen_id']=np.asarray(specimen_ids).astype(str)
    if filenames is not None:
        payload['filename']=np.asarray(filenames).astype(str)
    np.savez_compressed(pdir/f'{tag}.npz', **payload)
    if history is not None:
        hdir=Path(out_dir)/'histories'; hdir.mkdir(parents=True, exist_ok=True)
        json.dump(history, open(hdir/f'{tag}.json','w'))
    ece,bins = expected_calibration_error(np.asarray(y_true), np.asarray(y_prob))
    cdir=Path(out_dir)/'calibration'; cdir.mkdir(parents=True, exist_ok=True)
    json.dump({'tag':tag,'ece':ece,'bins':bins,'classes':classes,**(extra or {})},
              open(cdir/f'{tag}.json','w'), indent=2)
    return ece


In [7]:
# ===== Load =====
man=pd.read_csv(SPLITS_DIR/'manifest_full.csv')
# ===== Rebuild image paths for THIS session =====
# manifest_full.csv stores absolute paths from the machine that ran NB01. Kaggle mounts
# datasets at different locations per session/account, so those paths are not portable.
# Rebuild them from RAW_DATASET_DIR + label + filename, then verify every file exists.
man['filepath'] = [str(Path(RAW_DATASET_DIR)/l/f) for l,f in zip(man.label, man.filename)]
missing = [p for p in man.filepath if not Path(p).exists()]
if missing:
    print(f'*** {len(missing)} of {len(man)} images NOT found. First few:')
    for p in missing[:5]: print('   ', p)
    print('\nFix RAW_DATASET_DIR above. It must be the folder that directly contains the')
    print('11 class sub-folders. Check the exact mount with:')
    print("   import os; print(os.listdir('/kaggle/input'))")
    raise FileNotFoundError(f'{len(missing)} images missing - stop here, do not train.')
print(f'OK - all {len(man)} image paths resolve in this session.')

CLASSES=sorted(man.label.unique()); NC=len(CLASSES)
label2idx={c:i for i,c in enumerate(CLASSES)}
pool=man[man.reserve=='pool']
trn=pool[pool.group_split=='train']; val=pool[pool.group_split=='val']
common_test=man[man.reserve=='common_test']
print('train',len(trn),'val',len(val),'common_test',len(common_test))

OK - all 11094 image paths resolve in this session.
train 6625 val 1397 common_test 1665


In [8]:
# ===== 1. Colour-histogram kNN, per class =====
_knn_cache = (Path(PREV_RUN_DIR)/'knn_color_per_class.json') if PREV_RUN_DIR else None
if _knn_cache is not None and _knn_cache.exists():
    knn_res=json.load(open(_knn_cache))
    json.dump(knn_res,open(OUT_DIR/'knn_color_per_class.json','w'),indent=2)
    print('reused cached colour-kNN result from previous run')
    print(f"colour-only kNN on common_test: acc={knn_res['accuracy']:.4f} f1={knn_res['f1_macro']:.4f}")
    print('\nper-class F1 (colour alone):')
    for k,v in sorted(knn_res['per_class_f1'].items(), key=lambda kv:-kv[1]): print(f'  {v:.3f}  {k}')
else:
    import cv2
    from sklearn.neighbors import KNeighborsClassifier

    def hsv_hist(p,bins=32):
        im=cv2.imread(p); im=cv2.resize(im,(128,128)); im=cv2.cvtColor(im,cv2.COLOR_BGR2HSV)
        h=[cv2.calcHist([im],[c],None,[bins],[0,256]).flatten() for c in range(3)]
        v=np.concatenate(h); return v/ (v.sum()+1e-8)

    def feats(df):
        return np.stack([hsv_hist(p) for p in df.filepath]), df.label.map(label2idx).values

    print('extracting colour features...')
    Xtr,ytr = feats(trn); Xte,yte = feats(common_test)
    knn=KNeighborsClassifier(n_neighbors=5).fit(Xtr,ytr)
    yp=knn.predict(Xte)

    acc=accuracy_score(yte,yp); f1m=f1_score(yte,yp,average='macro',zero_division=0)
    per=f1_score(yte,yp,average=None,labels=list(range(NC)),zero_division=0)
    knn_res={'accuracy':float(acc),'f1_macro':float(f1m),
             'per_class_f1':{CLASSES[i]:float(per[i]) for i in range(NC)},
             'n_neighbors':5,'feature':'HSV histogram 32 bins','eval_set':'common_test'}
    json.dump(knn_res,open(OUT_DIR/'knn_color_per_class.json','w'),indent=2)
    print(f'colour-only kNN on common_test: acc={acc:.4f} f1={f1m:.4f} (chance={1/NC:.3f})')
    print('\nper-class F1 (colour alone) - high values = class is solvable by colour:')
    for c,v in sorted(knn_res['per_class_f1'].items(), key=lambda kv:-kv[1]):
        print(f'  {v:.3f}  {c}')

extracting colour features...
colour-only kNN on common_test: acc=0.6721 f1=0.6690 (chance=0.091)

per-class F1 (colour alone) - high values = class is solvable by colour:
  0.951  Fungal Stripe Damage
  0.938  Dried Leaf
  0.766  Yellow Mosaic Virus
  0.736  Black Spot
  0.693  Pest-Affected Dry Leaf
  0.662  White Spot
  0.627  Burned Leaf
  0.569  Insect Chewing Damage
  0.568  Leaf Blight Disease
  0.541  Healthy Leaf
  0.308  Red Rust Disease


In [9]:
# ===== 2. Grayscale-trained CNN (how much does the CNN rely on colour?) =====
_,isz,bs = build_backbone(GRAY_MODEL,NC,pretrained=False)   # always needed downstream
_gray_cache = (Path(PREV_RUN_DIR)/'grayscale_ablation.json') if PREV_RUN_DIR else None

if _gray_cache is not None and _gray_cache.exists():
    _c=json.load(open(_gray_cache)); gray_m=_c['grayscale']; colour_m=_c.get('colour')
    json.dump(_c,open(OUT_DIR/'grayscale_ablation.json','w'),indent=2,default=float)
    print('reused cached grayscale/colour result from previous run')
    print(f'grayscale-trained: acc={gray_m["accuracy"]:.4f} f1={gray_m["f1_macro"]:.4f}')
    if colour_m:
        print(f'colour-trained  : acc={colour_m["accuracy"]:.4f} f1={colour_m["f1_macro"]:.4f}')
        print(f'DROP from removing colour: {colour_m["accuracy"]-gray_m["accuracy"]:+.4f} acc')
else:
    print('Training',GRAY_MODEL,'on GRAYSCALE images...')
    gm,gh,gbe = train_model(GRAY_MODEL,trn,val,label2idx,seed=SEED,epochs=EPOCHS,
                            patience=PATIENCE,grayscale=True)
    y,p,pr = predict(gm,common_test,label2idx,isz,bs,grayscale=True)
    gray_m = metrics_dict(y,p,pr,NC)
    gray_m['ece']=save_run_artifacts(OUT_DIR,f'{GRAY_MODEL}_grayscale',y,p,pr,history=gh,
                                     classes=CLASSES,extra={'set':'common_test','variant':'grayscale'},
                                     specimen_ids=common_test.specimen_id.values,
                                     filenames=common_test.filename.values)
    torch.save(gm.state_dict(), OUT_DIR/f'{GRAY_MODEL}_grayscale_seed{SEED}.pt')
    print(f'grayscale-trained: acc={gray_m["accuracy"]:.4f} f1={gray_m["f1_macro"]:.4f}')
    del gm; torch.cuda.empty_cache()

    colour_m=None
    if EVAL_CKPT is not None and EVAL_CKPT.exists():
        cm,_,_=build_backbone(GRAY_MODEL,NC); cm.load_state_dict(torch.load(EVAL_CKPT,map_location=device))
        cm=cm.to(device).eval()
        y,p,pr=predict(cm,common_test,label2idx,isz,bs)
        colour_m=metrics_dict(y,p,pr,NC)
        colour_m['ece']=save_run_artifacts(OUT_DIR,f'{GRAY_MODEL}_colour_ref',y,p,pr,
                                     classes=CLASSES,extra={'set':'common_test','variant':'colour'},
                                     specimen_ids=common_test.specimen_id.values,
                                     filenames=common_test.filename.values)
        print(f'colour-trained  : acc={colour_m["accuracy"]:.4f} f1={colour_m["f1_macro"]:.4f}')
        print(f'DROP from removing colour: {colour_m["accuracy"]-gray_m["accuracy"]:+.4f} acc')
        del cm; torch.cuda.empty_cache()
    else:
        print('No colour reference checkpoint - colour-vs-grayscale comparison skipped.')
    json.dump({'grayscale':gray_m,'colour':colour_m},
              open(OUT_DIR/'grayscale_ablation.json','w'),indent=2,default=float)

Training ResNet50 on GRAYSCALE images...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 183MB/s]


  ep 01/30 train_loss=0.5118 acc=0.8415 | val_loss=0.7850 acc=0.7645
  ep 02/30 train_loss=0.1281 acc=0.9632 | val_loss=0.5546 acc=0.8325
  ep 03/30 train_loss=0.0876 acc=0.9754 | val_loss=0.9336 acc=0.7845
  ep 04/30 train_loss=0.0741 acc=0.9784 | val_loss=0.2781 acc=0.9134
  ep 05/30 train_loss=0.0642 acc=0.9792 | val_loss=0.3617 acc=0.8776
  ep 06/30 train_loss=0.0547 acc=0.9835 | val_loss=0.2412 acc=0.9234
  ep 07/30 train_loss=0.0510 acc=0.9842 | val_loss=0.2999 acc=0.9112
  ep 08/30 train_loss=0.0404 acc=0.9885 | val_loss=0.5799 acc=0.8640
  ep 09/30 train_loss=0.0476 acc=0.9852 | val_loss=0.2040 acc=0.9334
  ep 10/30 train_loss=0.0269 acc=0.9926 | val_loss=0.3972 acc=0.8840
  ep 11/30 train_loss=0.0207 acc=0.9941 | val_loss=0.1639 acc=0.9477
  ep 12/30 train_loss=0.0277 acc=0.9920 | val_loss=0.4133 acc=0.9034
  ep 13/30 train_loss=0.0224 acc=0.9931 | val_loss=0.1184 acc=0.9592
  ep 14/30 train_loss=0.0088 acc=0.9979 | val_loss=0.1400 acc=0.9492
  ep 15/30 train_loss=0.0132 acc=0

In [11]:
# ===== Summary =====
print('='*64)
print('COLOUR SHORTCUT SUMMARY'); print('='*64)
print(f'colour-only kNN (common_test)   acc = {knn_res["accuracy"]:.4f}')
print(f'grayscale-trained CNN           acc = {gray_m["accuracy"]:.4f}')
if colour_m: print(f'colour-trained CNN              acc = {colour_m["accuracy"]:.4f}')
print('\nExternal validation (concern #5) is in NB11, not here.')
print('Saved to',OUT_DIR)


SHORTCUT / EXTERNAL SUMMARY
colour-only kNN (common_test)   acc = 0.6721
grayscale-trained CNN           acc = 0.9580
colour-trained CNN              acc = 0.9850
zero-shot external              acc = 0.4591

Saved to /kaggle/working/revision_shortcut
